# Laboratorio 5 - Task 1

**Contexto:** control de semáforos en una intersección con flujo variable. Esta entrega es solo de análisis; no se implementa código.

**Supuestos.** Cuatro carriles de entrada, dos fases y dos acciones: mantener o cambiar. Las variables continuas están normalizadas a $[0,1]$.

**Uso responsable de IA.** La IA se usó para ordenar las ideas. Prompt: *"Para un laboratorio de aprendizaje por refuerzo, propone una estructura rigurosa para comparar aproximación lineal y tabular en control de semáforos, sin implementar código."* Funciona porque delimita el dominio y la comparación.

## 1. Vector de características

Para cada carril $j\in\{1,2,3,4\}$: densidad $\rho_j$, velocidad promedio $v_j$ y espera acumulada $\tau_j$. Se agregan la fase actual $p_0,p_1$ y un sesgo.

$$\mathbf{x}(s)=[1,\rho_1,v_1,\tau_1,\ldots,\rho_4,v_4,\tau_4,p_0,p_1]^T\in\mathbb{R}^{15}.$$

| Componente | Rango | Por qué se incluye |
|---|---:|---|
| $1$ (sesgo) | $\{1\}$ | Permite un valor base aun cuando las demás variables sean cero. |
| $\rho_j$, para cada carril | $[0,1]$ | Resume congestión y anticipa la formación de cola. |
| $v_j$, para cada carril | $[0,1]$ | Distingue una vía densa pero móvil de una vía bloqueada. |
| $\tau_j$, para cada carril | $[0,1]$ | Representa el costo de equidad: no dejar un carril esperando demasiado tiempo. |
| $p_0,p_1$ | $\{0,1\}$, con $p_0+p_1=1$ | El mismo tráfico tiene distinto valor según qué movimiento tenga luz verde. |

El vector cubre congestión, flujo, espera y fase. La forma lineal $\hat V(s;\mathbf w)=\mathbf w^T\mathbf x(s)$ no captura interacciones por sí sola. Ejemplos necesarios: $\rho_j(1-v_j)$, $\tau_j(1-p_{\text{verde},j})$ y diferencias de espera entre carriles.

## 2. Cantidad de parámetros y comparación tabular

Para aproximar $V(s)$ con el vector anterior hay **15 parámetros**: uno por característica. Si se requiere aproximar $Q(s,a)$ con dos acciones, la forma lineal usual usa un bloque de pesos por acción; entonces tendría $2\times15=30$ parámetros.

La tabla $Q$ debe distinguir las 12 variables continuas (tres por cada uno de los cuatro carriles), la fase y la acción. Con 10 niveles por variable:

$$|Q|=10^{12}\times2\text{ fases}\times2\text{ acciones}=4\times10^{12}\text{ entradas}. $$

La aproximación lineal usa 30 pesos frente a cuatro billones de entradas. La tabla necesita visitar cada combinación y no comparte aprendizaje entre estados cercanos. La aproximación lineal sí generaliza entre estados similares, pero puede no capturar relaciones no lineales.

La tabla puede ajustar ruido en celdas poco visitadas. El modelo lineal tiene menor riesgo de sobreajuste, pero mayor riesgo de subajuste.

## 3. Por qué TD semi-gradiente llega al punto de TD

Con referencia fija, $J(\mathbf w)=\mathbb E[(V_\pi(S)-\hat V(S;\mathbf w))^2]$. TD usa $R_{t+1}+\gamma\hat V(S_{t+1};\mathbf w)$, que depende de $\mathbf w$.

$$\mathbf w\leftarrow\mathbf w+\alpha\,[R_{t+1}+\gamma\hat V(S_{t+1};\mathbf w)-\hat V(S_t;\mathbf w)]\,\mathbf x(S_t)$$

deriva solo respecto a $\hat V(S_t;\mathbf w)$ e ignora la derivada de la referencia. No minimiza el gradiente completo de $J(\mathbf w)$; converge al punto de TD, que puede diferir del mínimo global.

El error es más grave con $\gamma$ cercano a 1, porque se propaga a más pasos. Para el cruce se recomienda probar $\gamma\in[0.90,0.95]$ y validar espera, equidad y estabilidad.

## 4. Triada mortal y dictamen inicial

La triada mortal combina aproximación de función, *bootstrapping* y aprendizaje fuera de política. El componente más relevante es el **bootstrapping**: una estimación incorrecta de congestión futura entra en la meta TD y afecta decisiones posteriores. El aprendizaje fuera de política importa si se entrena con una política y se evalúa otra.

Deep RL no está justificado automáticamente. Para un cruce aislado, la aproximación lineal con rasgos de interacción puede ser suficiente. Deep RL se justificaría si aparecen relaciones no lineales persistentes, varias intersecciones conectadas o efectos temporales que el vector no captura.